In [1]:
# import dependencies
import numpy as np
import pandas as pd
import re    # removes symbols, numbers, punctuation
from nltk.corpus import stopwords       # removes useless words like "the", "is", "and", "a"
from nltk.stem.porter import PorterStemmer  # reduces words to root form "running" → "run"
from sklearn.feature_extraction.text import TfidfVectorizer # converts text into numbers the model can understand
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

d:\download 99\Anaconda\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
d:\download 99\Anaconda\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [7]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\yousef\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

In [ ]:
# printing the stopwords in english
print(stopwords.words('english'))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

In [9]:
# load_data
data = pd.read_csv('data.csv')
data.head()

,id,title,author,text,label
0,0,House Dem Aide: We Didn’t Even See Comey’s Let...,Darrell Lucus,House Dem Aide: We Didn’t Even See Comey’s Let...,1
1,1,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",Daniel J. Flynn,Ever get the feeling your life circles the rou...,0
2,2,Why the Truth Might Get You Fired,Consortiumnews.com,"Why the Truth Might Get You Fired October 29, ...",1
3,3,15 Civilians Killed In Single US Airstrike Hav...,Jessica Purkiss,Videos 15 Civilians Killed In Single US Airstr...,1
4,4,Iranian woman jailed for fictional unpublished...,Howard Portnoy,Print \nAn Iranian woman has been sentenced to...,1


In [11]:
print('-----------------------------')
print(data.shape)
print('-----------------------------')
print(data.isnull().sum())


-----------------------------
(20800, 5)
-----------------------------
id           0
title      558
author    1957
text        39
label        0
dtype: int64


In [13]:
# replacing the missing values with empty string
data = data.fillna('')

In [20]:
# merging the author and news title
data['content'] = data['author']+" "+ data['title']
data.head()

,id,title,author,text,label,content
0,0,House Dem Aide: We Didn’t Even See Comey’s Let...,Darrell Lucus,House Dem Aide: We Didn’t Even See Comey’s Let...,1,Darrell Lucus House Dem Aide: We Didn’t Even S...
1,1,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",Daniel J. Flynn,Ever get the feeling your life circles the rou...,0,"Daniel J. Flynn FLYNN: Hillary Clinton, Big Wo..."
2,2,Why the Truth Might Get You Fired,Consortiumnews.com,"Why the Truth Might Get You Fired October 29, ...",1,Consortiumnews.com Why the Truth Might Get You...
3,3,15 Civilians Killed In Single US Airstrike Hav...,Jessica Purkiss,Videos 15 Civilians Killed In Single US Airstr...,1,Jessica Purkiss 15 Civilians Killed In Single ...
4,4,Iranian woman jailed for fictional unpublished...,Howard Portnoy,Print \nAn Iranian woman has been sentenced to...,1,Howard Portnoy Iranian woman jailed for fictio...


In [40]:
# step1: stemming 
# stemming: is the process of reducing a word to its root word
# actor = actress = acting ---> act
port_stem = PorterStemmer()

def stemming(content):  # content example --> "The Actors are Acting in a Theatrical Performance!!!"
    stemmed_content = re.sub('[^a-zA-Z]',' ', content) # remove any thing that is not letter ---> "The Actors are Acting in a Theatrical Performance   "
    stemmed_content = stemmed_content.lower()   # convert everything to a lowercase          ---> "the actors are acting in a theatrical performance"
    stemmed_content = stemmed_content.split()   # convert the string into list of words      ---> ["the", "actors", "are", "acting", "in", "a", "theatrical", "performance"]
    stemmed_content = [port_stem.stem(word) for word in stemmed_content if not word in stopwords.words('english')]  # Remove Stopwords + Stem    ---> ["actor", "act", "theatric", "perform"]
    stemmed_content = ' '.join(stemmed_content) # convert the list back to string   ---> "actor act theatric perform"0
    return stemmed_content

data['content'] = data['content'].apply(stemming)


In [41]:
data['content']

0        darrel lucu hou dem aid even see comey letter ...
1        daniel j flynn flynn hillari clinton big woman...
2                   consortiumnew com truth might get fire
3        jessica purkiss civilian kill singl us airstri...
4        howard portnoy iranian woman jail fiction unpu...
                               ...                        
20795    jerom hudson rapper trump poster child white s...
20796    benjamin hoffman n f l playoff schedul matchup...
20797    michael j de la merc rachel abram maci said re...
20798    alex ansari nato russia hold parallel exerci b...
20799                            david swanson keep f aliv
Name: content, Length: 20800, dtype: str

In [ ]:
# seperating the data & label
x = data['content'].values
y = data['label'].values

# converting the textual data to numerical data 
vectorizer = TfidfVectorizer()
vectorizer.fit(x)
x=vectorizer.transform(x)

print(x)


  (0, 263)	0.2701012497770876
  (0, 2464)	0.36765196867972083
  (0, 2933)	0.24684501285337127
  (0, 3567)	0.3598939188262558
  (0, 3759)	0.27053324808454915
  (0, 4917)	0.23331696690935097
  (0, 6930)	0.2187416908935914
  (0, 7612)	0.24785219520671598
  (0, 8546)	0.2921251408704368
  (0, 8822)	0.36359638063260746
  (0, 13348)	0.2565896679337956
  (0, 15553)	0.2848506356272864
  (1, 1481)	0.2957471154505952
  (1, 1877)	0.15614790568229528
  (1, 2206)	0.36915639258038363
  (1, 2790)	0.19208753385709676
  (1, 3535)	0.2653147533915268
  (1, 5440)	0.7186013955384664
  (1, 6744)	0.19152496072048605
  (1, 16656)	0.3025156488372128
  (2, 2917)	0.3179886800654691
  (2, 3072)	0.46097489583229645
  (2, 5326)	0.3866530551182615
  (2, 5903)	0.3474613386728292
  (2, 9532)	0.49351492943649944
  :	:
  (20797, 3610)	0.211655450844435
  (20797, 6969)	0.21809398920480086
  (20797, 8282)	0.22333184464489023
  (20797, 8901)	0.3617803783368178
  (20797, 9431)	0.2939494781564304
  (20797, 9500)	0.17463635692

In [48]:
# split data
x_train , x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,stratify=y,random_state=2)

In [ ]:
# traing the model
model = LogisticRegression().fit(x_train,y_train)

In [51]:
# evaluation
train_prediction = model.predict(x_train)
train_accuracy = accuracy_score(train_prediction,y_train)
print("training accuracy: ", train_accuracy)
# evaluation
test_prediction = model.predict(x_test)
test_accuracy = accuracy_score(test_prediction,y_test)
print("training accuracy: ", test_accuracy)


training accuracy:  0.9863581730769231
training accuracy:  0.9790865384615385


In [55]:
x_new = x_test[1]
prediction = model.predict(x_new)
print(prediction)

if prediction[0] == 1:
    print("Fake news")
else:
    print("Real news")



[0]
Real news


In [57]:
x_new = x_test[1]
prediction = model.predict(x_new)

if prediction[0] == 1:
    print("Fake News 🚨")
else:
    print("Real News ✅")



Real News ✅


In [58]:
print(y_test[1])

0
